In [ ]:

import pandas as pd
import hashlib
import json
from pathlib import Path

# Find the project root
root = Path.cwd()
if not (root / "data/modeling/modeling_dataset.parquet").exists():
    root = root.parent

# Verify dataset integrity
path = root / "data/modeling/modeling_dataset.parquet"
manifest_path = root / "data/modeling/manifest.json"

manifest = json.loads(manifest_path.read_text())
expected_hash = manifest["outputs_sha256"][path.name]
actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()

assert actual_hash == expected_hash, "Dataset hash mismatch!"

# Load data
df = pd.read_parquet(path)

print("Dataset shape:", df.shape)
print("\nData partitions:")
print(df["partition"].value_counts())

print("\nFirst 5 rows:")
display(df[["station_id", "timestamp", "pickups", "dropoffs"]].head())


In [ ]:

# Explore targets and weather features

print("Available columns:")
for col in df.columns:
    print(col)

print("\nStations:", df["station_id"].nunique())

print("\nMissing target values:")
print(df[["pickups", "dropoffs"]].isna().sum())

print("\nWeather variables:")
weather_cols = [
    "temperature",
    "wind_speed",
    "precipitation"
]

for col in weather_cols:
    if col in df.columns:
        print(f"{col}: {df[col].isna().sum()} missing values")
    else:
        print(f"{col}: column not found")

print("\nAverage hourly pickups by station:")
display(
    df.groupby("station_id")["pickups"]
      .agg(["count", "mean", "std", "max"])
      .round(2)
)


In [ ]:

#Prepare station-level forecasting data

station_id = "6197.08"

station_df = (
    df.loc[df["station_id"].astype(str) == station_id]
      .copy()
      .sort_values("timestamp")
)

print("Station:", station_id)
print("Total observations:", len(station_df))

print("\nData partitions:")
print(station_df["partition"].value_counts())

print("\nMissing values:")
print(
    station_df[
        ["pickups", "dropoffs",
         "temperature", "wind_speed",
         "precipitation"]
    ].isna().sum()
)

print("\nFirst 5 observations:")
display(
    station_df[
        ["timestamp", "pickups", "dropoffs",
         "temperature", "wind_speed",
         "precipitation", "partition"]
    ].head()
)


In [ ]:

# Define forecasting target and time order

import numpy as np

model_df = station_df.copy()

# Preserve unique chronological instants across DST
model_df["timestamp_utc"] = (
    model_df["timestamp"].dt.tz_convert("UTC")
)

model_df = (
    model_df.sort_values("timestamp_utc")
            .reset_index(drop=True)
)

# Forecasting setup
target = "pickups"
forecast_horizon = 1

# One-hour lag: observed demand from the preceding hour
model_df["pickups_lag1"] = model_df[target].shift(1)

# Known-ahead calendar predictors
local_time = model_df["timestamp"]

model_df["hour_sin"] = np.sin(
    2 * np.pi * local_time.dt.hour / 24
)
model_df["hour_cos"] = np.cos(
    2 * np.pi * local_time.dt.hour / 24
)

model_df["weekday_sin"] = np.sin(
    2 * np.pi * local_time.dt.dayofweek / 7
)
model_df["weekday_cos"] = np.cos(
    2 * np.pi * local_time.dt.dayofweek / 7
)

print("Forecast horizon:", forecast_horizon, "hour")
print("Target:", target)

print("\nPrepared features:")
display(
    model_df[
        ["timestamp", "pickups", "pickups_lag1",
         "hour_sin", "hour_cos",
         "weekday_sin", "weekday_cos",
         "partition"]
    ].head(8)
)

print("\nMissing lagged observations:",
      model_df["pickups_lag1"].isna().sum())


In [ ]:

# Prepare chronological train/validation sets

features = [
    "pickups_lag1",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos"
]

# Keep original data unchanged
train_df = model_df.loc[
    model_df["partition"] == "initial_train"
].copy()

val_df = model_df.loc[
    model_df["partition"] == "validation"
].copy()

# Record eligible observations without altering targets
train_ready = train_df.dropna(
    subset=["pickups"] + features
).copy()

val_ready = val_df.dropna(
    subset=["pickups"] + features
).copy()

# Numerical features for model fitting
X_train = train_ready[features].astype(float)
y_train = train_ready["pickups"].astype(float)

X_val = val_ready[features].astype(float)
y_val = val_ready["pickups"].astype(float)

print("Training rows:", len(train_ready))
print("Validation rows:", len(val_ready))

print("\nTraining target statistics:")
print(y_train.describe())

print("\nValidation target statistics:")
print(y_val.describe())


In [ ]:

# Fit an initial ARIMAX model

import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Known-ahead calendar predictors
exog_cols = [
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos"
]

# Keep the complete hourly training grid
train_full = model_df.loc[
    model_df["partition"] == "initial_train"
].copy().reset_index(drop=True)

y = train_full["pickups"].astype(float)
X = train_full[exog_cols].astype(float)

# AR(1) errors + calendar regressors
arimax_model = SARIMAX(
    endog=y,
    exog=X,
    order=(1, 0, 0),
    trend="c",
    enforce_stationarity=True,
    enforce_invertibility=True
)

arimax_result = arimax_model.fit(
    disp=False,
    maxiter=100
)

print("Converged:",
      arimax_result.mle_retvals.get("converged"))

print("AIC:", round(arimax_result.aic, 2))
print("BIC:", round(arimax_result.bic, 2))

print("\nModel coefficients:")
print(arimax_result.params.round(4))


In [ ]:

# Fixed rolling one-hour-ahead validation

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

val_full = model_df.loc[
    model_df["partition"] == "validation"
].copy()

# The validation index must continue after the training index
n_train = len(train_full)
val_full.index = pd.RangeIndex(
    start=n_train,
    stop=n_train + len(val_full)
)

y_val_full = val_full["pickups"].astype(float)
X_val_full = val_full[exog_cols].astype(float)

# Extend the fitted model without re-estimating parameters
rolling_result = arimax_result.extend(
    endog=y_val_full,
    exog=X_val_full
)

# One-step-ahead forecasts using previous observed data
pred = rolling_result.get_prediction(
    information_set="predicted"
).predicted_mean

validation_results = val_full[
    ["station_id", "timestamp", "pickups", "partition"]
].copy()

validation_results["predicted_pickups"] = np.clip(
    np.asarray(pred, dtype=float), 0, None
)

# Exclude missing targets from evaluation
valid = (
    validation_results["pickups"].notna()
    & validation_results["predicted_pickups"].notna()
)

actual = validation_results.loc[valid, "pickups"].astype(float)
forecast = validation_results.loc[valid, "predicted_pickups"]

mae = mean_absolute_error(actual, forecast)
rmse = np.sqrt(mean_squared_error(actual, forecast))

print("Station:", station_id)
print("Forecast horizon: 1 hour")
print("Scored hours:", valid.sum())
print("Excluded hours:", (~valid).sum())
print("Validation MAE:", round(mae, 3))
print("Validation RMSE:", round(rmse, 3))

display(validation_results.head(10))


In [ ]:

# Visualize actual vs. predicted pickups

import matplotlib.pyplot as plt

# Select January 6–12, 2025 (New York local time)
plot_df = validation_results.loc[
    (validation_results["timestamp"] >= pd.Timestamp(
        "2025-01-06", tz="America/New_York"
    )) &
    (validation_results["timestamp"] < pd.Timestamp(
        "2025-01-13", tz="America/New_York"
    ))
].copy()

plt.figure(figsize=(14, 5))

plt.plot(
    plot_df["timestamp"],
    plot_df["pickups"],
    label="Actual Pickups",
    linewidth=1.5
)

plt.plot(
    plot_df["timestamp"],
    plot_df["predicted_pickups"],
    label="ARIMAX Prediction",
    linewidth=1.5,
    linestyle="--"
)

plt.title(
    "Citi Bike Hourly Pickup Forecast — Station 6197.08"
)
plt.xlabel("Date")
plt.ylabel("Number of Pickups per Hour")
plt.legend()
plt.grid(alpha=0.25)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:

# Create lagged weather features

weather_df = model_df.copy()

# Lag weather observations by one hour
weather_df["temperature_lag1"] = (
    weather_df["temperature"].shift(1)
)

weather_df["wind_speed_lag1"] = (
    weather_df["wind_speed"].shift(1)
)

print("Weather feature missing values:")
print(
    weather_df[
        ["temperature_lag1", "wind_speed_lag1"]
    ].isna().sum()
)

print("\nFirst observations:")
display(
    weather_df[
        ["timestamp", "pickups",
         "temperature", "temperature_lag1",
         "wind_speed", "wind_speed_lag1",
         "partition"]
    ].head(10)
)

print("\nMissing weather features by partition:")
print(
    weather_df.groupby("partition")[
        ["temperature_lag1", "wind_speed_lag1"]
    ].apply(lambda x: x.isna().sum())
)


In [ ]:

# Prepare weather features without future leakage

weather_model_df = weather_df.copy()

weather_features = [
    "temperature_lag1",
    "wind_speed_lag1"
]

# Mark originally missing lagged observations
for col in weather_features:
    weather_model_df[f"{col}_missing"] = (
        weather_model_df[col].isna().astype(int)
    )

# Only use recent past observations (maximum 3 rows)
for col in weather_features:
    weather_model_df[col] = (
        weather_model_df[col].ffill(limit=3)
    )

# Check remaining missing values
print("Remaining missing weather values:")
print(weather_model_df[weather_features].isna().sum())

print("\nWeather missingness by partition:")
print(
    weather_model_df.groupby("partition", observed=True)[
        weather_features
    ].apply(lambda x: x.isna().sum())
)

print("\nFirst 8 rows:")
display(
    weather_model_df[
        ["timestamp"] + weather_features +
        [f"{c}_missing" for c in weather_features]
    ].head(8)
)


In [ ]:

# Train Weather-Enhanced ARIMAX

import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

weather_exog_cols = [
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "temperature_lag1",
    "wind_speed_lag1",
    "temperature_lag1_missing",
    "wind_speed_lag1_missing"
]

weather_train = weather_model_df.loc[
    weather_model_df["partition"] == "initial_train"
].copy().reset_index(drop=True)

weather_val = weather_model_df.loc[
    weather_model_df["partition"] == "validation"
].copy().reset_index(drop=True)

# Fit fallback imputation values using TRAINING DATA ONLY
weather_medians = weather_train[
    ["temperature_lag1", "wind_speed_lag1"]
].median()

for col in ["temperature_lag1", "wind_speed_lag1"]:
    weather_train[col] = weather_train[col].fillna(
        weather_medians[col]
    )
    weather_val[col] = weather_val[col].fillna(
        weather_medians[col]
    )

X_weather_train = weather_train[
    weather_exog_cols
].astype(float)

y_weather_train = weather_train[
    "pickups"
].astype(float)

weather_arimax_model = SARIMAX(
    endog=y_weather_train,
    exog=X_weather_train,
    order=(1, 0, 0),
    trend="c",
    enforce_stationarity=True,
    enforce_invertibility=True
)

weather_arimax_result = weather_arimax_model.fit(
    disp=False,
    maxiter=100
)

print("Converged:",
      weather_arimax_result.mle_retvals.get("converged"))

print("AIC:", round(weather_arimax_result.aic, 2))
print("BIC:", round(weather_arimax_result.bic, 2))

print("\nWeather coefficients:")
print(
    weather_arimax_result.params[
        ["temperature_lag1", "wind_speed_lag1"]
    ].round(4)
)


In [ ]:

# Weather ARIMAX validation

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Continue validation index after training
n_train = len(weather_train)

weather_val.index = pd.RangeIndex(
    start=n_train,
    stop=n_train + len(weather_val)
)

X_weather_val = weather_val[
    weather_exog_cols
].astype(float)

y_weather_val = weather_val[
    "pickups"
].astype(float)

# Update states using observed validation targets,
# without re-estimating model parameters
weather_rolling = weather_arimax_result.extend(
    endog=y_weather_val,
    exog=X_weather_val
)

weather_pred = weather_rolling.get_prediction(
    information_set="predicted"
).predicted_mean

weather_validation_results = weather_val[
    ["station_id", "timestamp", "pickups"]
].copy()

weather_validation_results["weather_prediction"] = np.clip(
    np.asarray(weather_pred, dtype=float),
    0,
    None
)

# Align forecasts from both models by timestamp
comparison = validation_results[
    ["timestamp", "pickups", "predicted_pickups"]
].merge(
    weather_validation_results[
        ["timestamp", "weather_prediction"]
    ],
    on="timestamp",
    how="inner",
    validate="one_to_one"
)

# Evaluate on the SAME valid hours
valid = comparison[
    ["pickups", "predicted_pickups", "weather_prediction"]
].notna().all(axis=1)

scored = comparison.loc[valid].copy()

actual = scored["pickups"].astype(float)
base_pred = scored["predicted_pickups"].astype(float)
weather_pred = scored["weather_prediction"].astype(float)

base_mae = mean_absolute_error(actual, base_pred)
base_rmse = np.sqrt(mean_squared_error(actual, base_pred))

weather_mae = mean_absolute_error(actual, weather_pred)
weather_rmse = np.sqrt(mean_squared_error(actual, weather_pred))

print("Station:", station_id)
print("Scored hours:", len(scored))
print("Excluded hours:", len(comparison) - len(scored))

print("\nCalendar ARIMAX")
print("MAE:", round(base_mae, 3))
print("RMSE:", round(base_rmse, 3))

print("\nWeather ARIMAX")
print("MAE:", round(weather_mae, 3))
print("RMSE:", round(weather_rmse, 3))

print("\nMAE improvement (%):",
      round(100 * (base_mae - weather_mae) / base_mae, 2))

print("RMSE improvement (%):",
      round(100 * (base_rmse - weather_rmse) / base_rmse, 2))


In [ ]:

# Compare model predictions visually

import matplotlib.pyplot as plt

comparison_plot = scored.loc[
    (scored["timestamp"] >= pd.Timestamp(
        "2025-01-06", tz="America/New_York"
    )) &
    (scored["timestamp"] < pd.Timestamp(
        "2025-01-13", tz="America/New_York"
    ))
].copy()

plt.figure(figsize=(14, 5))

plt.plot(
    comparison_plot["timestamp"],
    comparison_plot["pickups"],
    label="Actual Pickups",
    linewidth=1.8
)

plt.plot(
    comparison_plot["timestamp"],
    comparison_plot["predicted_pickups"],
    label="Calendar ARIMAX",
    linestyle="--",
    linewidth=1.4
)

plt.plot(
    comparison_plot["timestamp"],
    comparison_plot["weather_prediction"],
    label="Weather ARIMAX",
    linestyle=":",
    linewidth=1.8
)

plt.title(
    "Citi Bike Forecast Comparison - Station 6197.08"
)
plt.xlabel("Date")
plt.ylabel("Hourly Pickups")
plt.legend()
plt.grid(alpha=0.25)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:

# Analyze forecast errors

error_df = scored.copy()

error_df["calendar_abs_error"] = (
    error_df["pickups"] -
    error_df["predicted_pickups"]
).abs()

error_df["weather_abs_error"] = (
    error_df["pickups"] -
    error_df["weather_prediction"]
).abs()

error_df["weather_extra_error"] = (
    error_df["weather_abs_error"] -
    error_df["calendar_abs_error"]
)

print("Top 10 hours where Weather ARIMAX performs worse:")
display(
    error_df.sort_values(
        "weather_extra_error", ascending=False
    )[
        ["timestamp", "pickups",
         "predicted_pickups", "weather_prediction",
         "calendar_abs_error", "weather_abs_error",
         "weather_extra_error"]
    ].head(10).round(2)
)

print("\nTop 10 hours where Weather ARIMAX performs better:")
display(
    error_df.sort_values(
        "weather_extra_error"
    )[
        ["timestamp", "pickups",
         "predicted_pickups", "weather_prediction",
         "weather_extra_error"]
    ].head(10).round(2)
)


In [ ]:

# Diagnose extreme Weather ARIMAX errors

# Find the 10 largest Weather ARIMAX absolute errors
worst_hours = error_df.nlargest(
    10, "weather_abs_error"
)[["timestamp"]].copy()

# Join with original weather features
diagnostic = worst_hours.merge(
    weather_model_df[
        [
            "timestamp",
            "temperature",
            "wind_speed",
            "temperature_lag1",
            "wind_speed_lag1",
            "temperature_lag1_missing",
            "wind_speed_lag1_missing"
        ]
    ],
    on="timestamp",
    how="left",
    validate="one_to_one"
)

# Join prediction and actual values
diagnostic = diagnostic.merge(
    error_df[
        [
            "timestamp",
            "pickups",
            "predicted_pickups",
            "weather_prediction",
            "weather_abs_error"
        ]
    ],
    on="timestamp",
    how="left",
    validate="one_to_one"
)

display(diagnostic)


In [ ]:

# Add annual seasonality features

import numpy as np

seasonal_df = weather_model_df.copy()

# Day of year based on New York local time
day_of_year = seasonal_df["timestamp"].dt.dayofyear

# Annual Fourier features
seasonal_df["year_sin"] = np.sin(
    2 * np.pi * day_of_year / 365.25
)

seasonal_df["year_cos"] = np.cos(
    2 * np.pi * day_of_year / 365.25
)

print("Annual seasonality features created!")

display(
    seasonal_df[
        ["timestamp", "hour_sin", "hour_cos",
         "year_sin", "year_cos", "partition"]
    ].head(10)
)

print("\nMissing annual features:")
print(
    seasonal_df[["year_sin", "year_cos"]]
    .isna().sum()
)


In [ ]:

# Fit Weather ARIMAX with Annual Seasonality

import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

seasonal_exog_cols = [
    "hour_sin", "hour_cos",
    "weekday_sin", "weekday_cos",
    "year_sin", "year_cos",
    "temperature_lag1", "wind_speed_lag1",
    "temperature_lag1_missing",
    "wind_speed_lag1_missing"
]

seasonal_train = seasonal_df.loc[
    seasonal_df["partition"] == "initial_train"
].copy().reset_index(drop=True)

seasonal_val = seasonal_df.loc[
    seasonal_df["partition"] == "validation"
].copy()

# Apply existing training-only fallback values
for col in ["temperature_lag1", "wind_speed_lag1"]:
    seasonal_train[col] = seasonal_train[col].fillna(weather_medians[col])
    seasonal_val[col] = seasonal_val[col].fillna(weather_medians[col])

X_seasonal_train = seasonal_train[seasonal_exog_cols].astype(float)
y_seasonal_train = seasonal_train["pickups"].astype(float)

seasonal_model = SARIMAX(
    endog=y_seasonal_train,
    exog=X_seasonal_train,
    order=(1, 0, 0),
    trend="c",
    enforce_stationarity=True,
    enforce_invertibility=True
)

seasonal_result = seasonal_model.fit(
    disp=False,
    maxiter=100
)

print("Converged:", seasonal_result.mle_retvals.get("converged"))
print("AIC:", round(seasonal_result.aic, 2))
print("BIC:", round(seasonal_result.bic, 2))

print("\nSelected coefficients:")
print(
    seasonal_result.params[
        ["year_sin", "year_cos",
         "temperature_lag1", "wind_speed_lag1"]
    ].round(4)
)


In [ ]:

# Validate Weather + Annual ARIMAX

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Continue validation index after training
n_train = len(seasonal_train)

seasonal_val.index = pd.RangeIndex(
    start=n_train,
    stop=n_train + len(seasonal_val)
)

X_seasonal_val = seasonal_val[
    seasonal_exog_cols
].astype(float)

y_seasonal_val = seasonal_val[
    "pickups"
].astype(float)

# Rolling one-step predictions without parameter refitting
seasonal_rolling = seasonal_result.extend(
    endog=y_seasonal_val,
    exog=X_seasonal_val
)

seasonal_pred = seasonal_rolling.get_prediction(
    information_set="predicted"
).predicted_mean

seasonal_validation_results = seasonal_val[
    ["timestamp", "pickups"]
].copy()

seasonal_validation_results["seasonal_prediction"] = np.clip(
    np.asarray(seasonal_pred, dtype=float),
    0,
    None
)

# Align all three models by timestamp
comparison_three = comparison.merge(
    seasonal_validation_results[
        ["timestamp", "seasonal_prediction"]
    ],
    on="timestamp",
    how="inner",
    validate="one_to_one"
)

# Use exactly the same scored hours
model_cols = [
    "pickups",
    "predicted_pickups",
    "weather_prediction",
    "seasonal_prediction"
]

scored_three = comparison_three.dropna(
    subset=model_cols
).copy()

actual = scored_three["pickups"].astype(float)

results = []

for model_name, col in [
    ("Calendar ARIMAX", "predicted_pickups"),
    ("Weather ARIMAX", "weather_prediction"),
    ("Weather + Annual ARIMAX", "seasonal_prediction")
]:
    pred = scored_three[col].astype(float)

    results.append({
        "Model": model_name,
        "MAE": mean_absolute_error(actual, pred),
        "RMSE": np.sqrt(mean_squared_error(actual, pred))
    })

metrics_table = pd.DataFrame(results)

print("Station:", station_id)
print("Forecast horizon: 1 hour")
print("Scored hours:", len(scored_three))
print("Excluded hours:", len(comparison_three) - len(scored_three))

display(metrics_table.round(3))


In [ ]:

# Check all stations before scaling models

station_summary = (
    df.groupby(["station_id", "partition"], observed=True)
      .agg(
          total_hours=("timestamp", "size"),
          valid_pickups=("pickups", "count"),
          mean_pickups=("pickups", "mean"),
          missing_temperature=("temperature",
                               lambda x: x.isna().sum()),
          missing_wind=("wind_speed",
                        lambda x: x.isna().sum())
      )
      .reset_index()
)

print("Station coverage summary:")
display(station_summary.round(2))

print("\nTotal stations:", df["station_id"].nunique())
print("Total rows:", len(df))


In [ ]:

# Fit Calendar ARIMAX for a second station

import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

second_station = "5184.08"

station2 = (
    df.loc[df["station_id"].astype(str) == second_station]
      .copy()
      .sort_values("timestamp")
      .reset_index(drop=True)
)

station2["hour_sin"] = np.sin(
    2 * np.pi * station2["timestamp"].dt.hour / 24
)
station2["hour_cos"] = np.cos(
    2 * np.pi * station2["timestamp"].dt.hour / 24
)
station2["weekday_sin"] = np.sin(
    2 * np.pi * station2["timestamp"].dt.dayofweek / 7
)
station2["weekday_cos"] = np.cos(
    2 * np.pi * station2["timestamp"].dt.dayofweek / 7
)

cols = ["hour_sin", "hour_cos", "weekday_sin", "weekday_cos"]

train2 = station2.loc[
    station2["partition"] == "initial_train"
].copy()

val2 = station2.loc[
    station2["partition"] == "validation"
].copy()

model2 = SARIMAX(
    endog=train2["pickups"].astype(float),
    exog=train2[cols].astype(float),
    order=(1, 0, 0),
    trend="c",
    enforce_stationarity=True,
    enforce_invertibility=True
)

fit2 = model2.fit(disp=False, maxiter=100)

print("Station:", second_station)
print("Converged:", fit2.mle_retvals.get("converged"))
print("AIC:", round(fit2.aic, 2))


In [ ]:

# Reusable station-level ARIMAX function

import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

calendar_features = [
    "hour_sin", "hour_cos",
    "weekday_sin", "weekday_cos"
]

def evaluate_station_arimax(data, station_id):

    # Select one station
    station = (
        data.loc[data["station_id"].astype(str) == str(station_id)]
        .copy()
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    # Calendar features
    local_time = station["timestamp"]

    station["hour_sin"] = np.sin(
        2 * np.pi * local_time.dt.hour / 24
    )
    station["hour_cos"] = np.cos(
        2 * np.pi * local_time.dt.hour / 24
    )
    station["weekday_sin"] = np.sin(
        2 * np.pi * local_time.dt.dayofweek / 7
    )
    station["weekday_cos"] = np.cos(
        2 * np.pi * local_time.dt.dayofweek / 7
    )

    train = station.loc[
        station["partition"] == "initial_train"
    ].copy().reset_index(drop=True)

    val = station.loc[
        station["partition"] == "validation"
    ].copy()

    # Preserve continuous chronological index
    val.index = pd.RangeIndex(
        len(train), len(train) + len(val)
    )

    # Train model
    model = SARIMAX(
        endog=train["pickups"].astype(float),
        exog=train[calendar_features].astype(float),
        order=(1, 0, 0),
        trend="c",
        enforce_stationarity=True,
        enforce_invertibility=True
    )

    fitted = model.fit(disp=False, maxiter=100)

    # One-step-ahead rolling predictions
    rolling = fitted.extend(
        endog=val["pickups"].astype(float),
        exog=val[calendar_features].astype(float)
    )

    pred = rolling.get_prediction(
        information_set="predicted"
    ).predicted_mean

    predictions = np.clip(
        np.asarray(pred, dtype=float), 0, None
    )

    actual = val["pickups"].to_numpy(dtype=float)

    valid = np.isfinite(actual) & np.isfinite(predictions)

    mae = mean_absolute_error(
        actual[valid], predictions[valid]
    )

    rmse = np.sqrt(mean_squared_error(
        actual[valid], predictions[valid]
    ))

    return {
        "station_id": str(station_id),
        "converged": fitted.mle_retvals.get("converged"),
        "MAE": round(mae, 3),
        "RMSE": round(rmse, 3),
        "scored_hours": int(valid.sum()),
        "excluded_hours": int((~valid).sum())
    }

print("Reusable ARIMAX function created!")


In [ ]:

# Test reusable ARIMAX function

test_result = evaluate_station_arimax(
    df,
    "6197.08"
)

print("ARIMAX Function Test Results:")
display(pd.DataFrame([test_result]))


In [ ]:

# Evaluate Calendar ARIMAX across all 8 stations

all_results = []

station_ids = sorted(
    df["station_id"].astype(str).unique()
)

for sid in station_ids:
    print(f"Training station {sid}...")

    try:
        result = evaluate_station_arimax(df, sid)
        all_results.append(result)

        print(
            f"  MAE: {result['MAE']:.3f} | "
            f"RMSE: {result['RMSE']:.3f} | "
            f"Converged: {result['converged']}"
        )

    except Exception as e:
        print(f"  FAILED: {sid} - {e}")
        raise

all_station_metrics = pd.DataFrame(all_results)

print("\nAll-station Validation Results:")
display(all_station_metrics)

print("\nAverage MAE:",
      round(all_station_metrics["MAE"].mean(), 3))

print("Average RMSE:",
      round(all_station_metrics["RMSE"].mean(), 3))


In [ ]:

# Review station-level forecasting performance

assert len(all_station_metrics) == 8, "Expected 8 stations"
assert all_station_metrics["converged"].eq(True).all(), \
    "Some models did not converge"

station_ranking = (
    all_station_metrics
    .sort_values("MAE")
    .reset_index(drop=True)
)

print("Station ranking by Validation MAE:")
display(station_ranking)

print("\nBest station:")
print(station_ranking.iloc[0][["station_id", "MAE", "RMSE"]])

print("\nMost challenging station:")
print(station_ranking.iloc[-1][["station_id", "MAE", "RMSE"]])


In [ ]:

# Define reusable Weather ARIMAX function

import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_weather_arimax(data, station_id):

    station = (
        data.loc[data["station_id"].astype(str) == str(station_id)]
        .copy()
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    t = station["timestamp"]

    station["hour_sin"] = np.sin(2*np.pi*t.dt.hour/24)
    station["hour_cos"] = np.cos(2*np.pi*t.dt.hour/24)
    station["weekday_sin"] = np.sin(2*np.pi*t.dt.dayofweek/7)
    station["weekday_cos"] = np.cos(2*np.pi*t.dt.dayofweek/7)

    # Previous-hour observed weather
    weather_cols = ["temperature", "wind_speed"]

    for col in weather_cols:
        lag_col = f"{col}_lag1"
        station[lag_col] = station[col].shift(1)
        station[f"{lag_col}_missing"] = (
            station[lag_col].isna().astype(int)
        )
        station[lag_col] = station[lag_col].ffill(limit=3)

    predictors = [
        "hour_sin", "hour_cos",
        "weekday_sin", "weekday_cos",
        "temperature_lag1", "wind_speed_lag1",
        "temperature_lag1_missing",
        "wind_speed_lag1_missing"
    ]

    train = station.loc[
        station["partition"] == "initial_train"
    ].copy().reset_index(drop=True)

    val = station.loc[
        station["partition"] == "validation"
    ].copy()

    # Training-only fallback imputation
    for col in ["temperature_lag1", "wind_speed_lag1"]:
        median = train[col].median()
        train[col] = train[col].fillna(median)
        val[col] = val[col].fillna(median)

    val.index = pd.RangeIndex(
        len(train), len(train) + len(val)
    )

    fit = SARIMAX(
        endog=train["pickups"].astype(float),
        exog=train[predictors].astype(float),
        order=(1, 0, 0),
        trend="c",
        enforce_stationarity=True,
        enforce_invertibility=True
    ).fit(disp=False, maxiter=100)

    rolling = fit.extend(
        endog=val["pickups"].astype(float),
        exog=val[predictors].astype(float)
    )

    pred = np.clip(
        np.asarray(
            rolling.get_prediction(
                information_set="predicted"
            ).predicted_mean,
            dtype=float
        ),
        0, None
    )

    actual = val["pickups"].to_numpy(dtype=float)
    valid = np.isfinite(actual) & np.isfinite(pred)

    return {
        "station_id": str(station_id),
        "converged": fit.mle_retvals.get("converged"),
        "MAE": round(mean_absolute_error(
            actual[valid], pred[valid]
        ), 3),
        "RMSE": round(np.sqrt(mean_squared_error(
            actual[valid], pred[valid]
        )), 3),
        "scored_hours": int(valid.sum()),
        "excluded_hours": int((~valid).sum())
    }

print("Reusable Weather ARIMAX function created!")


In [ ]:

#  Test reusable Weather ARIMAX function

weather_test_result = evaluate_weather_arimax(
    df,
    "6197.08"
)

print("Weather ARIMAX Function Test:")
display(pd.DataFrame([weather_test_result]))

# Compare with the previous single-station result
print("\nPreviously calculated results:")
print("MAE: 5.568")
print("RMSE: 8.401")


In [ ]:

# Evaluate Weather ARIMAX across all 8 stations

weather_results = []

station_ids = sorted(df["station_id"].astype(str).unique())

for sid in station_ids:
    print(f"Training Weather ARIMAX for station {sid}...", flush=True)

    result = evaluate_weather_arimax(df, sid)
    weather_results.append(result)

    print(
        f"MAE: {result['MAE']:.3f} | "
        f"RMSE: {result['RMSE']:.3f} | "
        f"Converged: {result['converged']}",
        flush=True
    )

weather_station_metrics = pd.DataFrame(weather_results)

assert len(weather_station_metrics) == 8, "Missing station results"
assert weather_station_metrics["converged"].eq(True).all(), \
    "Some models did not converge"

print("\nAll-station Weather ARIMAX Results:")
display(weather_station_metrics)

print("\nAverage MAE:",
      round(weather_station_metrics["MAE"].mean(), 3))

print("Average RMSE:",
      round(weather_station_metrics["RMSE"].mean(), 3))


In [ ]:

# Compare Calendar vs Weather ARIMAX

comparison_all = all_station_metrics.merge(
    weather_station_metrics,
    on="station_id",
    suffixes=("_calendar", "_weather"),
    validate="one_to_one"
)

# Check matching evaluation coverage
assert (
    comparison_all["scored_hours_calendar"]
    == comparison_all["scored_hours_weather"]
).all(), "Scored-hour counts do not match"

comparison_all["MAE_improvement_%"] = (
    100 * (
        comparison_all["MAE_calendar"]
        - comparison_all["MAE_weather"]
    ) / comparison_all["MAE_calendar"]
)

comparison_all["RMSE_improvement_%"] = (
    100 * (
        comparison_all["RMSE_calendar"]
        - comparison_all["RMSE_weather"]
    ) / comparison_all["RMSE_calendar"]
)

final_comparison = comparison_all[
    [
        "station_id",
        "MAE_calendar",
        "MAE_weather",
        "MAE_improvement_%",
        "RMSE_calendar",
        "RMSE_weather",
        "RMSE_improvement_%"
    ]
].sort_values("MAE_improvement_%", ascending=False)

display(final_comparison.round(2))

print(
    "Stations improved on MAE:",
    (final_comparison["MAE_improvement_%"] > 0).sum(),
    "/ 8"
)

print(
    "Stations improved on RMSE:",
    (final_comparison["RMSE_improvement_%"] > 0).sum(),
    "/ 8"
)


In [ ]:

# Export validation comparison results

from pathlib import Path

output_dir = root / "results" / "arimax"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "station_model_comparison_2025.csv"

final_comparison.to_csv(output_file, index=False)

print("Saved comparison table to:")
print(output_file)

print("\nRows:", len(final_comparison))
print("Columns:", len(final_comparison.columns))
